In [44]:
q1 = "I just discovered the course, can I still join?"

q2 = "I just found out about the program, can I still enroll?"

In [45]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [46]:
v1 = model.encode(q1)
v1.shape

(384,)

In [47]:
v2 = model.encode(q2)
v2.shape

(384,)

In [48]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [49]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [50]:
v1.dot(dv)

np.float32(0.32332402)

In [51]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [52]:
v2.dot(dv)

np.float32(0.019730477)

In [53]:
from ingest import load_faq_data

documents = load_faq_data()

In [54]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [55]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/30 [00:00<?, ?it/s]

1470

In [56]:
import numpy as np
X = np.array(vectors)

In [57]:
scores = X.dot(v1)

In [58]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(1074), np.float32(0.762941))

In [59]:
documents[1074]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [60]:
top5 = np.argsort(scores)[-5:]

In [61]:
top5

array([1079,    0,  190,  732, 1074])

In [62]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.5600999
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date,

In [63]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [64]:
results = vindex.search(
    v1,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [65]:
from dotenv import load_dotenv
from openai import OpenAI
from os import getenv

load_dotenv()
openai_client = OpenAI(
    base_url="https://26120057-2478-resource.openai.azure.com/openai/v1",
    api_key=getenv("AZURE_API_KEY"),
)

In [66]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [67]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [68]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can sign up. If you want to receive a certificate, you’ll need to submit your project while submissions are still open.'

In [69]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [70]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [ ]:
vector_assistant.rag("the program has already begun, can I still sign up?")